In [44]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from sir_graph.dynamics import simulate, run_ensemble
from sir_graph.emm import search_subgroups
from sir_graph.geometric import random_geometric
from sir_graph.havel_hakimi import (
    havel_hakimi,
    randomize_degree_preserving,
)
from sir_graph.parameters import NodeParams

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
N_NODES = 1000
DATA_SEED = 22

people = pd.read_csv("../data/synthetic_data.csv")

people = (
    people
    .sample(n=N_NODES, random_state=DATA_SEED)
    .reset_index(drop=True)
)

people.insert(
    0,
    "node_id",
    np.arange(N_NODES, dtype=np.int64)
)

infection_rate = people[
    "infection_rate"
].to_numpy(dtype=np.float64)

people.head()

,node_id,person_id,age,sex,occupation,smoking,alcohol,obesity,infection_rate
0,0,35220,62,Female,Student,0,0,0,0.3329
1,1,29487,71,Female,Office-based,0,0,0,0.3428
2,2,38566,40,Male,Agriculture,1,1,1,0.3998
3,3,20529,70,Male,Office-based,1,1,0,0.4563
4,4,30182,54,Male,Healthcare,0,0,0,0.3793


In [46]:
from sir_graph.geometric import random_geometric

N_NODES = 1000
RADIUS = 0.0723
NETWORK_SEED = 22

baseline_graph, positions = random_geometric(
    N_NODES,
    RADIUS,
    rng=NETWORK_SEED,
)

print("Nodes:", baseline_graph.n_nodes)
print("Edges:", baseline_graph.n_edges)
print("Mean degree:", baseline_graph.degree().mean())

Nodes: 1000
Edges: 7723
Mean degree: 15.446


In [47]:
original_degree_sequence = baseline_graph.degree().copy()

print("Mean degree:", original_degree_sequence.mean())
print("Min degree:", original_degree_sequence.min())
print("Max degree:", original_degree_sequence.max())

Mean degree: 15.446
Min degree: 2
Max degree: 27


In [48]:
from sir_graph.havel_hakimi import (
    havel_hakimi,
    randomize_degree_preserving,
)

hh_graph = havel_hakimi(original_degree_sequence)

In [49]:
import numpy as np
print("HH nodes:", hh_graph.n_nodes)
print("HH edges:", hh_graph.n_edges)
print("HH mean degree:", hh_graph.degree().mean())

print(
    "Degrees identical:",
    np.array_equal(
        hh_graph.degree(),
        original_degree_sequence
    )
)

HH nodes: 1000
HH edges: 7723
HH mean degree: 15.446
Degrees identical: True


In [50]:
N_SWAPS = 10 * hh_graph.n_edges

random_graph = randomize_degree_preserving(
    hh_graph,
    n_swaps=N_SWAPS,
    rng=22,
)

In [51]:
print("Randomized nodes:", random_graph.n_nodes)
print("Randomized edges:", random_graph.n_edges)
print("Randomized mean degree:", random_graph.degree().mean())

print(
    "Degrees still identical:",
    np.array_equal(
        random_graph.degree(),
        original_degree_sequence
    )
)

Randomized nodes: 1000
Randomized edges: 7723
Randomized mean degree: 15.446
Degrees still identical: True


In [52]:
graph = random_graph

In [53]:
planted_mask = (
    (people["age"] >= 60)
    & (people["smoking"] == 1)
).to_numpy()

print("Planted subgroup size:", planted_mask.sum())
print("Fraction of population:", planted_mask.mean())

Planted subgroup size: 117
Fraction of population: 0.117


In [54]:
from sir_graph.parameters import NodeParams

RECOVERY_PROB = 0.4

base_params = NodeParams(
    susceptibility=infection_rate,
    infectiousness=np.ones(N_NODES),
    recovery=np.full(N_NODES, RECOVERY_PROB),
)

In [55]:
PLANTED_MULTIPLIER = 1.75

planted_params = base_params.scaled(
    planted_mask,
    infectiousness=PLANTED_MULTIPLIER,
)

In [56]:
print(
    "Normal infectiousness:",
    np.unique(planted_params.infectiousness[~planted_mask])
)

print(
    "Planted infectiousness:",
    np.unique(planted_params.infectiousness[planted_mask])
)

Normal infectiousness: [1.]
Planted infectiousness: [1.75]


In [57]:
print(
    "Susceptibility unchanged:",
    np.array_equal(
        base_params.susceptibility,
        planted_params.susceptibility
    )
)

Susceptibility unchanged: True


In [58]:
P_VALUES = [0.08, 0.10, 0.12, 0.14, 0.15]

comparison = []

for p_value in P_VALUES:
    for condition, current_params in [
        ("control", base_params),
        ("planted", planted_params),
    ]:
        test_runs = run_ensemble(
            graph,
            n_runs=100,
            n_seeds=5,
            p=p_value,
            params=current_params,
            rng=24,
        )

        final_sizes = np.array([
            r.final_size
            for r in test_runs
        ])

        comparison.append({
            "condition": condition,
            "p": p_value,
            "mean_final_size": final_sizes.mean(),
            "mean_fraction_infected": final_sizes.mean() / N_NODES,
            "median_final_size": np.median(final_sizes),
            "min_final_size": final_sizes.min(),
            "max_final_size": final_sizes.max(),
        })

comparison_df = pd.DataFrame(comparison)

comparison_df

,condition,p,mean_final_size,mean_fraction_infected,median_final_size,min_final_size,max_final_size
0,control,0.08,59.78,0.05978,39.0,5,269
1,planted,0.08,105.85,0.10585,51.5,5,333
2,control,0.10,247.74,0.24774,296.5,5,511
3,planted,0.10,386.95,0.38695,440.5,8,558
4,control,0.12,479.26,0.47926,533.5,6,644
5,planted,0.12,600.92,0.60092,615.5,8,708
6,control,0.14,634.26,0.63426,657.5,8,728
7,planted,0.14,713.46,0.71346,715.5,623,768
8,control,0.15,696.70,0.69670,700.0,14,766
9,planted,0.15,754.74,0.75474,761.5,686,801


In [59]:


P = 0.12
N_RUNS = 500
N_SEEDS = 5

# Separate reproducible random streams
seed_rng = np.random.default_rng(1001)
epidemic_rng = np.random.default_rng(1002)

# Same initial infected people for control and planted condition
seed_sets = [
    np.sort(
        seed_rng.choice(
            N_NODES,
            size=N_SEEDS,
            replace=False
        )
    )
    for _ in range(N_RUNS)
]

# Same simulation seed for corresponding control/planted runs
run_rng_seeds = epidemic_rng.integers(
    0,
    2**32 - 1,
    size=N_RUNS,
    dtype=np.uint32,
)

In [60]:
def run_fixed_ensemble(params):
    return [
        simulate(
            graph,
            seeds=seeds,
            p=P,
            params=params,
            rng=int(run_seed),
        )
        for seeds, run_seed in zip(
            seed_sets,
            run_rng_seeds
        )
    ]

In [61]:
control_runs = run_fixed_ensemble(base_params)
planted_runs = run_fixed_ensemble(planted_params)

print("Control runs:", len(control_runs))
print("Planted runs:", len(planted_runs))

Control runs: 500
Planted runs: 500


In [62]:
control_final_sizes = np.array([
    r.final_size for r in control_runs
])

planted_final_sizes = np.array([
    r.final_size for r in planted_runs
])

print(
    "Control mean final size:",
    control_final_sizes.mean()
)

print(
    "Planted mean final size:",
    planted_final_sizes.mean()
)

Control mean final size: 496.334
Planted mean final size: 581.396


In [63]:
def per_node_spreading_metrics(runs):
    infection_times = np.stack([
        r.infection_time
        for r in runs
    ])

    offspring = np.stack([
        r.offspring
        for r in runs
    ])

    # > 0 means naturally infected;
    # time 0 means artificially selected as seed
    naturally_infected = infection_times > 0

    n_natural_infections = naturally_infected.sum(axis=0)

    total_secondary = np.where(
        naturally_infected,
        offspring,
        0
    ).sum(axis=0)

    mean_secondary_when_infected = np.divide(
        total_secondary,
        n_natural_infections,
        out=np.full(N_NODES, np.nan),
        where=n_natural_infections > 0,
    )

    return (
        n_natural_infections,
        mean_secondary_when_infected
    )

In [64]:
(
    control_infection_count,
    control_spread
) = per_node_spreading_metrics(control_runs)

(
    planted_infection_count,
    planted_spread
) = per_node_spreading_metrics(planted_runs)

In [65]:
analysis_df = people.copy()

analysis_df["degree"] = graph.degree()

analysis_df["control_infection_count"] = (
    control_infection_count
)

analysis_df["control_mean_secondary"] = (
    control_spread
)

analysis_df["planted_infection_count"] = (
    planted_infection_count
)

analysis_df["planted_mean_secondary"] = (
    planted_spread
)

# Ground truth — ONLY for evaluation later
analysis_df["is_planted"] = planted_mask

In [66]:
analysis_df.groupby("is_planted")[
    [
        "degree",
        "control_mean_secondary",
        "planted_mean_secondary",
    ]
].mean()

,degree,control_mean_secondary,planted_mean_secondary
is_planted,,,
False,15.497169,0.928806,0.856661
True,15.059829,0.912526,1.413209


In [67]:
x = analysis_df["degree"].to_numpy()
y = analysis_df["planted_mean_secondary"].to_numpy()

valid = np.isfinite(y)

emm_slope, emm_intercept = np.polyfit(
    x[valid],
    y[valid],
    1,
)

analysis_df["emm_expected_spread"] = (
    emm_intercept
    + emm_slope * analysis_df["degree"]
)

analysis_df["emm_residual"] = (
    analysis_df["planted_mean_secondary"]
    - analysis_df["emm_expected_spread"]
)

print("EMM intercept:", emm_intercept)
print("EMM slope:", emm_slope)

EMM intercept: -0.10939657071224125
EMM slope: 0.06675991329425023


In [68]:
from sir_graph.emm import search_subgroups

CATEGORICAL_COLUMNS = [
    "sex",
    "occupation",
    "smoking",
    "alcohol",
    "obesity",
]

planted_results = search_subgroups(
    analysis_df,
    target_column="emm_residual",
    categorical_columns=CATEGORICAL_COLUMNS,
    age_column="age",
    age_thresholds=range(20, 81, 5),
    min_size=20,
    max_depth=2,
    direction="high",
)

planted_pairs = (
    planted_results[
        planted_results["depth"] == 2
    ]
    .reset_index(drop=True)
)

planted_pairs.head(20)

,depth,description,size,support,mean_residual,rest_mean_residual,difference,quality
0,2,smoking == 1 AND age >= 60,117,0.117,0.517212,-0.068532,0.585744,0.200355
1,2,smoking == 1 AND age >= 55,151,0.151,0.388422,-0.069083,0.457505,0.177780
2,2,smoking == 1 AND age >= 50,185,0.185,0.312274,-0.070884,0.383159,0.164803
3,2,obesity == 0 AND age >= 60,183,0.183,0.292663,-0.065554,0.358216,0.153240
4,2,smoking == 1 AND age >= 45,218,0.218,0.254380,-0.070914,0.325295,0.151882
5,2,smoking == 1 AND age >= 40,256,0.256,0.212699,-0.073187,0.285886,0.144648
6,2,smoking == 1 AND age >= 65,70,0.070,0.498543,-0.037525,0.536068,0.141830
7,2,smoking == 1 AND age >= 35,296,0.296,0.176275,-0.074116,0.250391,0.136228
8,2,obesity == 0 AND age >= 55,241,0.241,0.207087,-0.065755,0.272842,0.133943
9,2,smoking == 1 AND age >= 30,344,0.344,0.143761,-0.075387,0.219148,0.128534


In [69]:
analysis_df.groupby("is_planted").agg(
    count=("node_id", "count"),
    mean_degree=("degree", "mean"),
    mean_actual=("planted_mean_secondary", "mean"),
    mean_expected=("emm_expected_spread", "mean"),
    mean_residual=("emm_residual", "mean"),
)

,count,mean_degree,mean_actual,mean_expected,mean_residual
is_planted,,,,,
False,883,15.497169,0.856661,0.925193,-0.068532
True,117,15.059829,1.413209,0.895996,0.517212


In [70]:
discovered_mask = (
    (analysis_df["age"] >= 60)
    & (analysis_df["smoking"] == 1)
).to_numpy()

true_mask = analysis_df["is_planted"].to_numpy()

intersection = np.sum(discovered_mask & true_mask)
union = np.sum(discovered_mask | true_mask)

jaccard = intersection / union

print("Jaccard:", jaccard)

Jaccard: 1.0


In [71]:
x = analysis_df["degree"].to_numpy()
y = analysis_df["control_mean_secondary"].to_numpy()

valid = np.isfinite(y)

control_slope, control_intercept = np.polyfit(
    x[valid],
    y[valid],
    1,
)

analysis_df["control_expected"] = (
    control_intercept
    + control_slope * analysis_df["degree"]
)

analysis_df["control_residual"] = (
    analysis_df["control_mean_secondary"]
    - analysis_df["control_expected"]
)

print("Control intercept:", control_intercept)
print("Control slope:", control_slope)

Control intercept: -0.1079548708373312
Control slope: 0.06699831130236064


In [72]:
control_results = search_subgroups(
    analysis_df,
    target_column="control_residual",
    categorical_columns=CATEGORICAL_COLUMNS,
    age_column="age",
    age_thresholds=range(20, 81, 5),
    min_size=20,
    max_depth=2,
    direction="high",
)

control_pairs = (
    control_results[
        control_results["depth"] == 2
    ]
    .reset_index(drop=True)
)

control_pairs.head(20)

,depth,description,size,support,mean_residual,rest_mean_residual,difference,quality
0,2,smoking == 1 AND age >= 25,377,0.377,0.014542,-0.008800,0.023341,0.014332
1,2,smoking == 1 AND age >= 30,344,0.344,0.015829,-0.008301,0.024130,0.014153
2,2,smoking == 1 AND age < 80,599,0.599,0.007101,-0.010607,0.017708,0.013706
3,2,smoking == 1 AND age < 75,599,0.599,0.007101,-0.010607,0.017708,0.013706
4,2,smoking == 1 AND age >= 20,417,0.417,0.012314,-0.008808,0.021122,0.013640
5,2,smoking == 1 AND age >= 40,256,0.256,0.019822,-0.006821,0.026643,0.013480
6,2,smoking == 1 AND age >= 35,296,0.296,0.016285,-0.006847,0.023131,0.012585
7,2,smoking == 1 AND age < 70,563,0.563,0.006884,-0.008869,0.015753,0.011820
8,2,alcohol == 1 AND age >= 20,213,0.213,0.019026,-0.005149,0.024175,0.011157
9,2,smoking == 1 AND age < 65,529,0.529,0.007069,-0.007940,0.015009,0.010916


In [73]:
PLANTED_DESCRIPTION = "smoking == 1 AND age >= 60"

planted_rank = (
    planted_pairs.index[
        planted_pairs["description"] == PLANTED_DESCRIPTION
    ][0]
    + 1
)

control_rank = (
    control_pairs.index[
        control_pairs["description"] == PLANTED_DESCRIPTION
    ][0]
    + 1
)

print("Planted-condition rank:", planted_rank)
print("Control-condition rank:", control_rank)

display(
    planted_pairs[
        planted_pairs["description"] == PLANTED_DESCRIPTION
    ]
)

display(
    control_pairs[
        control_pairs["description"] == PLANTED_DESCRIPTION
    ]
)

Planted-condition rank: 1
Control-condition rank: 69


,depth,description,size,support,mean_residual,rest_mean_residual,difference,quality
0,2,smoking == 1 AND age >= 60,117,0.117,0.517212,-0.068532,0.585744,0.200355


,depth,description,size,support,mean_residual,rest_mean_residual,difference,quality
68,2,smoking == 1 AND age >= 60,117,0.117,0.011498,-0.001523,0.013021,0.004454
